# DIOR Faster R-CNN Training

Run training in Colab and download checkpoints and loss plots to your computer after each epoch.


In [ ]:
# Load the project source into the Colab runtime.
!test -d /content/ecs174-dior/.git || git clone https://github.com/viveknarayana/ecs174-dior.git /content/ecs174-dior
%cd /content/ecs174-dior
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# Pull the latest project updates before using the cloned source.
!git -C /content/ecs174-dior pull origin main


In [ ]:
# Download the DIOR archives and extract them into the directory structure expected by DIORDataset.
!pip -q install gdown matplotlib
!mkdir -p /content/dior_archives /content/DIOR
!gdown --folder 'https://drive.google.com/drive/folders/1UdlgHk49iu6WpcJ5467iT-UqNPpx__CC' -O /content/dior_archives

from pathlib import Path
from zipfile import ZipFile
archive_dir = Path('/content/dior_archives')
data_root = Path('/content/DIOR')
for archive in sorted(archive_dir.glob('*.zip')):
    print('Extracting', archive.name)
    with ZipFile(archive) as zf:
        zf.extractall(data_root)

assert (data_root / 'JPEGImages-trainval').is_dir(), 'JPEGImages-trainval folder not found after extraction'
assert (data_root / 'JPEGImages-test').is_dir(), 'JPEGImages-test folder not found after extraction'
assert (data_root / 'Annotations' / 'Horizontal Bounding Boxes').is_dir(), 'Annotations folder not found after extraction'
assert (data_root / 'ImageSets' / 'Main' / 'train.txt').is_file() or (data_root / 'Main' / 'train.txt').is_file(), 'train.txt split file not found after extraction'
print('DIOR folder layout looks ready:', data_root)

In [ ]:
# Run training and download completed epoch files through the browser.
import subprocess
import sys
import shutil
from datetime import datetime
from config import CONFIG
from pathlib import Path
from google.colab import files

OUTPUT_ROOT = '/content/dior_outputs'

def run_training(backbone, extra_args=None, output_root=OUTPUT_ROOT):
    command = [sys.executable, '-u', 'train.py', '--data-root', '/content/DIOR',
               '--backbone', backbone, '--output-root', output_root,
               '--epochs', '10', '--batch-size', '2'] + (extra_args or [])
    run_dir = Path(output_root) / backbone
    args = extra_args or []
    def option(name, default):
        return args[len(args) - 1 - args[::-1].index(name) + 1] if name in args else default
    mode = 'smoke' if '--tiny-test' in args else 'train'
    batch_size = option('--batch-size', 2)
    epochs = option('--epochs', 10)
    run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
    prefix = (f'{backbone}_{mode}_size{CONFIG.image_size}_batch{batch_size}'
              f'_lr{CONFIG.learning_rate}_epochs{epochs}_{run_id}')
    download_dir = run_dir / 'downloads'
    download_dir.mkdir(parents=True, exist_ok=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            if line.startswith('EPOCH_FILES_READY '):
                epoch = int(line.split()[1])
                for path in (run_dir / f'checkpoint_epoch_{epoch:02d}.pth',
                             run_dir / 'training_log.csv', run_dir / 'loss_curves.png'):
                    download_path = download_dir / f'{prefix}_epoch{epoch:02d}_{path.name}'
                    shutil.copy2(path, download_path)
                    print(f'Requesting download: {download_path.name}', flush=True)
                    files.download(str(download_path))
        if process.wait() != 0:
            raise RuntimeError('Training failed. Check the output above.')
    finally:
        if process.poll() is None:
            process.terminate()
            process.wait()


## Smoke test

Downloads the checkpoint, training log, and loss plot after one epoch on two training images and one validation image.


In [ ]:
run_training('resnet50', ['--tiny-test', '--tiny-train-images', '2', '--tiny-validation-images', '1', '--epochs', '1'], output_root='/content/smoke_test_outputs')


## Full experiments

Run one backbone at a time. Allow multiple downloads in your browser and check that each checkpoint finishes downloading. The test split is not used for training.


In [ ]:
# ResNet-50
run_training('resnet50')


In [ ]:
# MobileNetV3
run_training('mobilenet')


## Resume an interrupted run

Upload the latest checkpoint and its training_log.csv. Choose the same backbone and total epoch target as the original run.


In [ ]:
# Upload a checkpoint and training_log.csv to resume a ResNet-50 run.
from google.colab import files
import shutil
uploaded = files.upload()
checkpoints = [name for name in uploaded if name.endswith('.pth')]
logs = [name for name in uploaded if name.endswith('training_log.csv')]
if len(checkpoints) != 1 or len(logs) != 1:
    raise ValueError('Upload exactly one checkpoint and training_log.csv together.')
run_dir = Path(OUTPUT_ROOT) / 'resnet50'
run_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(logs[0], run_dir / 'training_log.csv')
run_training('resnet50', ['--resume', str(Path(checkpoints[0]).resolve())])
